# Lab 1 — K-Means Baseline

**Day 05 · Unsupervised Learning · Cisco AI/ML Training**

---

## Learning objectives

1. Aggregate **500** daily NYSE rows into **25** per-symbol feature vectors.
2. Scale features before distance-based clustering.
3. Fit **K-Means** with **k = 4** and interpret **inertia**.
4. Inspect cluster sizes — no labels, only structure in data.

> **Checkpoints:** **25** symbols · k = **4** · inertia ≈ **45.86** · sizes `{0:8, 1:9, 2:7, 3:1}`



## K-Means in one slide

Unlike Days 3–4 (predict `default`), today there is **no target label** — we discover groups.

| Step | What happens |
|------|--------------|
| 1. Aggregate | 500 daily rows → 25 symbol profiles |
| 2. Scale | `StandardScaler` — same reason as Day 4 KNN |
| 3. Cluster | K-Means assigns each symbol to one of **k** centroids |
| 4. Evaluate | **Inertia** = within-cluster sum of squared distances (lower = tighter) |

**Data path:** `500` daily OHLCV rows → `groupby("symbol")` → 4 numeric features per ticker.

---

## 1. Load NYSE daily data

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-05":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "nyse" / "nyse_stocks.csv").is_file():
            GH_ROOT = parent
            break

FEATURE_COLUMNS = ["avg_close", "volatility", "avg_volume", "avg_range"]

nyse = pd.read_csv(GH_ROOT / "data" / "nyse" / "nyse_stocks.csv", parse_dates=["date"])
print(f"daily rows: {len(nyse)}")
print(f"symbols: {nyse['symbol'].nunique()}")
display(nyse.head(3))

---

## 2. Build per-symbol features

In [ ]:
nyse["range"] = nyse["high"] - nyse["low"]
features = (
    nyse.groupby("symbol")
    .agg(
        avg_close=("close", "mean"),
        volatility=("close", "std"),
        avg_volume=("volume", "mean"),
        avg_range=("range", "mean"),
    )
    .reset_index()
)
features["volatility"] = features["volatility"].fillna(0.0)

print(f"symbols clustered: {len(features)}")
display(features.head(5))

| Feature | Meaning |
|---------|--------|
| `avg_close` | Mean closing price over the sample window |
| `volatility` | Std dev of close — price variability |
| `avg_volume` | Mean daily trading volume |
| `avg_range` | Mean daily high − low |

---

## 3. Scale features

In [ ]:
X = features[FEATURE_COLUMNS]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaled_df = pd.DataFrame(X_scaled, columns=FEATURE_COLUMNS)
print("Raw vs scaled (first symbol):")
display(pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "raw": X.iloc[0].round(2).values,
    "scaled": scaled_df.iloc[0].round(2).values,
}))

Without scaling, `avg_volume` (~25M) would dominate distance over `volatility` (~2–3).

---

## 4. Fit K-Means (k = 4)

In [ ]:
k = 4
model = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = model.fit_predict(X_scaled)

features = features.copy()
features["cluster"] = labels

print("Lab 1 — K-Means baseline")
print(f"features: {FEATURE_COLUMNS}")
print(f"k: {k}")
print(f"inertia: {model.inertia_:.4f}")

unique, counts = np.unique(labels, return_counts=True)
cluster_counts = dict(zip(unique.tolist(), counts.tolist()))
print(f"cluster counts: {cluster_counts}")

---

## 5. Cluster assignment table

In [ ]:
display(
    features[["symbol", "avg_close", "volatility", "cluster"]]
    .sort_values("cluster")
    .round(2)
)

Cluster **3** has only **1** symbol — K-Means can produce uneven segment sizes (see Lab 6 summary).

---

## 6. Visualize clusters (avg_close vs volatility)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=features,
    x="avg_close",
    y="volatility",
    hue="cluster",
    palette="tab10",
    s=100,
    ax=ax,
)
for _, row in features.iterrows():
    ax.annotate(row["symbol"], (row["avg_close"], row["volatility"]), fontsize=8, alpha=0.8)
ax.set_title(f"K-Means clusters (k={k}, inertia={model.inertia_:.1f})")
plt.tight_layout()
plt.show()

---

## 7. Extension — try k = 3 (preview Lab 2)

In [ ]:
model_k3 = KMeans(n_clusters=3, random_state=42, n_init=10)
model_k3.fit_predict(X_scaled)

compare_k = pd.DataFrame({
    "k": [3, 4],
    "inertia": [model_k3.inertia_, model.inertia_],
})
display(compare_k.round(4))
print("Lower inertia with more clusters — Lab 2 finds the best k via the elbow method.")

---

## 8. Checkpoint summary

In [ ]:
assert len(features) == 25
assert k == 4
assert abs(model.inertia_ - 45.8634) < 0.1
assert cluster_counts == {0: 8, 1: 9, 2: 7, 3: 1}
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. Why cluster **25 symbols** instead of all **500** daily rows?
2. What does lower inertia imply — and why doesn't it alone pick k?
3. How is this different from Day 4 KNN (supervised vs unsupervised)?

**Previous:** [Day 04 — Distance-Based ML](../day-04/README.md)  
**Next:** [Lab 2 — Elbow method](lab02_elbow_method.ipynb)